In [1]:
import pandas as pd

df = pd.read_csv("../data/traffic.csv")

# Force to numeric first, in case it's being read as string
df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')

# Now convert from Unix epoch seconds to datetime
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')

df = df.set_index('timestamp')

# fix numeric columns
df['port'] = pd.to_numeric(df['port'], errors='coerce')
df['size'] = pd.to_numeric(df['size'], errors='coerce')
print(df.head())
print(df.dtypes)

                                      src_ip         dst_ip protocol     port  \
timestamp                                                                       
2026-07-11 00:48:43.598185539  10.193.15.238  140.82.114.26      TCP    443.0   
2026-07-11 00:48:43.827745676  140.82.114.26  10.193.15.238      TCP  54200.0   
2026-07-11 00:48:44.349939823  10.193.15.238  10.193.15.237      UDP   1716.0   
2026-07-11 00:48:44.353423357  10.193.15.237  10.193.15.238     ICMP      NaN   
2026-07-11 00:48:44.418611050  10.193.15.238        8.8.8.8     ICMP      NaN   

                              flags  size  
timestamp                                  
2026-07-11 00:48:43.598185539     A    66  
2026-07-11 00:48:43.827745676     A    66  
2026-07-11 00:48:44.349939823   NaN  1514  
2026-07-11 00:48:44.353423357   NaN   590  
2026-07-11 00:48:44.418611050   NaN    98  
src_ip          str
dst_ip          str
protocol        str
port        float64
flags           str
size          int64
dtyp

In [2]:
# Resample into 10-second windows
windows = df.resample('10s')

# Starting with the simplest features
features = pd.DataFrame()
features['packet_count'] = windows.size()
features['avg_packet_size'] = windows['size'].mean()
features['unique_dst_ips'] = windows['dst_ip'].nunique()

print(features)

                     packet_count  avg_packet_size  unique_dst_ips
timestamp                                                         
2026-07-11 00:48:40            19       617.842105               5
2026-07-11 00:48:50             0              NaN               0
2026-07-11 00:49:00             0              NaN               0
2026-07-11 00:49:10             0              NaN               0
2026-07-11 00:49:20             0              NaN               0
...                           ...              ...             ...
2026-07-11 01:28:20             0              NaN               0
2026-07-11 01:28:30             0              NaN               0
2026-07-11 01:28:40             0              NaN               0
2026-07-11 01:28:50             1        66.000000               1
2026-07-11 01:29:00            37       617.918919               8

[243 rows x 3 columns]


In [3]:
active_windows = features[features['packet_count'] > 0]
print(active_windows)

                     packet_count  avg_packet_size  unique_dst_ips
timestamp                                                         
2026-07-11 00:48:40            19       617.842105               5
2026-07-11 01:28:50             1        66.000000               1
2026-07-11 01:29:00            37       617.918919               8


In [4]:
# floor each timestamp to it's 10-second window start
window_id = df.index.floor('10s')

# cross-tabulate: rows = window, columns = protocol, values = count
protocol_counts = pd.crosstab(window_id, df['protocol'], rownames=['timestamp'])

print(protocol_counts)

protocol             ICMP  TCP  UDP
timestamp                          
2026-07-11 00:48:40     3   15    1
2026-07-11 01:28:50     0    1    0
2026-07-11 01:29:00    10   21    6


In [5]:
protocol_pct = protocol_counts.div(protocol_counts.sum(axis=1), axis=0)
print(protocol_pct)

protocol                 ICMP       TCP       UDP
timestamp                                        
2026-07-11 00:48:40  0.157895  0.789474  0.052632
2026-07-11 01:28:50  0.000000  1.000000  0.000000
2026-07-11 01:29:00  0.270270  0.567568  0.162162


In [6]:
# boolean masks: does this row's flags contain S or A ?
df['has_syn'] = df['flags'].fillna('').str.contains('S')
df['has_ack'] = df['flags'].fillna('').str.contains('A')

# count SYNs and ACKs per window
syn_counts = df['has_syn'].groupby(window_id).sum()
ack_counts = df['has_ack'].groupby(window_id).sum()

# Ratio (avoid divide-by-zero)
syn_ack_ratio = syn_counts / ack_counts.replace(0, pd.NA)

print(syn_ack_ratio)


timestamp
2026-07-11 00:48:40    0.0
2026-07-11 01:28:50    0.0
2026-07-11 01:29:00    0.1
dtype: float64


In [7]:
from scipy.stats import entropy

def calc_port_entropy(group):
    port_counts = group['port'].value_counts()
    return entropy(port_counts, base=2)

port_entropy = df.groupby(window_id).apply(calc_port_entropy)
print(port_entropy)

timestamp
2026-07-11 00:48:40    1.423795
2026-07-11 01:28:50    0.000000
2026-07-11 01:29:00    1.826789
dtype: float64


In [8]:
final_features = active_windows.join(protocol_pct).join(syn_ack_ratio.rename('syn_ack_ratio')).join(port_entropy.rename('port_entropy'))
print(final_features)

                     packet_count  avg_packet_size  unique_dst_ips      ICMP  \
timestamp                                                                      
2026-07-11 00:48:40            19       617.842105               5  0.157895   
2026-07-11 01:28:50             1        66.000000               1  0.000000   
2026-07-11 01:29:00            37       617.918919               8  0.270270   

                          TCP       UDP  syn_ack_ratio  port_entropy  
timestamp                                                             
2026-07-11 00:48:40  0.789474  0.052632            0.0      1.423795  
2026-07-11 01:28:50  1.000000  0.000000            0.0      0.000000  
2026-07-11 01:29:00  0.567568  0.162162            0.1      1.826789  
